In [ ]:
import pandas as pd
import numpy as np
import csv

# =========================================================
# STEP 1: FIX BROKEN CSV STRUCTURE (QUOTE + COLUMN SHIFT)
# =========================================================

input_file = "triage.csv"
repaired_rows = []

with open(input_file, "r", encoding="utf-8", errors="replace") as f:
    reader = csv.reader(f)

    header = next(reader)
    repaired_rows.append(header)

    for row in reader:
        # expected schema = 11 columns
        if len(row) == 11:
            repaired_rows.append(row)
        else:
            # fix broken chiefcomplaint (merge extra columns)
            fixed = row[:10]
            fixed.append(",".join(row[10:]))
            repaired_rows.append(fixed)

# convert to dataframe
triage = pd.DataFrame(repaired_rows[1:], columns=repaired_rows[0])



Final shape: (425087, 11)
Null summary:
 subject_id            0
stay_id               0
temperature       23948
heartrate         17130
resprate          20413
o2sat             20688
sbp               18291
dbp               19091
pain              12896
acuity             6987
chiefcomplaint        0
dtype: int64
Pipeline completed successfully. Data is DB-safe.


In [ ]:
# =========================================================
# STEP 2: TYPE CLEANING
# =========================================================

num_cols = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity"]

for c in num_cols:
    triage[c] = pd.to_numeric(triage[c], errors="coerce")

triage["subject_id"] = pd.to_numeric(triage["subject_id"], errors="coerce")
triage["stay_id"] = pd.to_numeric(triage["stay_id"], errors="coerce")

In [ ]:


# =========================================================
# STEP 3: SCALE FIX (DATA CORRUPTION HANDLING)
# =========================================================

# Temperature (98.6 → 986)
triage.loc[triage["temperature"] > 200, "temperature"] /= 10

# O2 saturation scaling issues
triage.loc[triage["o2sat"] > 1000, "o2sat"] /= 10

# SBP extreme scaling
triage.loc[triage["sbp"] > 1000, "sbp"] /= 10
triage.loc[triage["sbp"] > 1000, "sbp"] /= 10

# DBP extreme scaling
triage.loc[triage["dbp"] > 10000, "dbp"] /= 1000
triage.loc[(triage["dbp"] > 1000) & (triage["dbp"] <= 10000), "dbp"] /= 100
triage.loc[(triage["dbp"] > 200) & (triage["dbp"] <= 1000), "dbp"] /= 10


In [ ]:

# =========================================================
# STEP 4: MEDICAL VALID RANGE FILTERING
# =========================================================

triage.loc[
    (triage["temperature"] < 85) | (triage["temperature"] > 110),
    "temperature"
] = np.nan

triage.loc[
    (triage["heartrate"] < 25) | (triage["heartrate"] > 250),
    "heartrate"
] = np.nan

triage.loc[
    (triage["resprate"] < 5) | (triage["resprate"] > 60),
    "resprate"
] = np.nan

triage.loc[
    (triage["o2sat"] < 50),
    "o2sat"
] = np.nan

triage["o2sat"] = triage["o2sat"].clip(upper=100)

In [ ]:

# =========================================================
# STEP 5: HARD DB SAFE CLIPPING (CRITICAL)
# =========================================================

triage["temperature"] = triage["temperature"].clip(85, 110)
triage["heartrate"] = triage["heartrate"].clip(25, 250)
triage["resprate"] = triage["resprate"].clip(5, 60)
triage["o2sat"] = triage["o2sat"].clip(50, 100)
triage["sbp"] = triage["sbp"].clip(60, 300)
triage["dbp"] = triage["dbp"].clip(30, 200)


In [ ]:


# =========================================================
# STEP 6: CLEAN TEXT
# =========================================================

triage["chiefcomplaint"] = triage["chiefcomplaint"].astype(str).str.replace('"', '', regex=False)

triage["pain"] = triage["pain"].astype(str).replace(["", "nan", "None"], np.nan)


In [ ]:

# =========================================================
# STEP 7: FINAL NULL NORMALIZATION
# =========================================================

triage = triage.replace([np.inf, -np.inf], np.nan)

# optional: drop rows without key identifiers
triage = triage.dropna(subset=["subject_id", "stay_id"])


In [ ]:

# =========================================================
# STEP 8: FINAL SANITY CHECK
# =========================================================

print("Final shape:", triage.shape)
print("Null summary:\n", triage.isna().sum())


In [ ]:

# =========================================================
# STEP 9: READY FOR POSTGRES INSERT
# =========================================================

triage.to_csv("triage_fixed.csv", index=False)

print("Pipeline completed successfully. Data is DB-safe.")

In [3]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="asclena_db",
    user="postgres",
    password="admin123"
)

cur = conn.cursor()

In [4]:
df = pd.read_csv("triage_fixed.csv")

df = df.replace({np.nan: None})

In [5]:
query = """
INSERT INTO asclena.triage (
    subject_id, stay_id, temperature,
    heartrate, resprate, o2sat,
    sbp, dbp, pain, acuity, chiefcomplaint
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

batch_size = 5000
failed_rows = []

for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]

    try:
        cur.executemany(query, batch.values.tolist())
        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Batch failed at {i}: {e}")

        # log bad batch for debugging
        failed_rows.append(i)